In [ ]:
import pandas as pd
import numpy as np 
from pathlib import Path
from sentence_transformers import SentenceTransformer

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "app").exists():
            return candidate
    raise FileNotFoundError("Could not locate the BTS Song Atlas project root.")


PROJECT_ROOT = find_project_root()


In [ ]:
BASE_DIR = PROJECT_ROOT
PROCESSED_DIR = BASE_DIR / "data" / "processed"

songs = pd.read_csv(PROCESSED_DIR / "song_master.csv")

songs_with_lyrics = songs[songs["lyrics_clean"].notna().copy()]

print(songs.shape)
print(songs_with_lyrics.shape)

In [ ]:
model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

In [ ]:
lyrics_texts = songs_with_lyrics["lyrics_clean"].tolist()

embeddings = model.encode(
    lyrics_texts,
    show_progress_bar=True, 
    convert_to_numpy=True,
    normalize_embeddings=True
)

embeddings.shape

In [ ]:
embedding_cols = [f"embedding_{i}" for i in range(embeddings.shape[1])]

embeddings_df = pd.DataFrame(
    embeddings,
    columns=embedding_cols
)

song_embeddings = pd.concat(
    [
        songs_with_lyrics[[
            "track_id", 
            "track_name",
            "album_name",
            "release_year"
        ]].reset_index(drop=True),
        embeddings_df
    ],
    axis=1
)

song_embeddings.head()

In [ ]:
song_embeddings.to_parquet(
    PROCESSED_DIR / "song_embeddings.parquet",
    index=False
)

print("Saved:", PROCESSED_DIR / "song_embeddings.parquet")
print("Shape:", song_embeddings.shape)

In [ ]:
check = pd.read_parquet(PROCESSED_DIR / "song_embeddings.parquet")

print(check.shape)
check.head()

In [ ]:
song_embeddings.isna().sum().sum()

In [ ]:
len([c for c in song_embeddings.columns if c.startswith("embedding_")])


In [ ]:
song_embeddings["track_id"].is_unique

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

black_swan = song_embeddings[
    song_embeddings["track_name"] == "Black Swan"
].iloc[0]

fake_love = song_embeddings[
    song_embeddings["track_name"] == "FAKE LOVE"
].iloc[0]

tomorrow = song_embeddings[
    song_embeddings["track_name"] == "Tomorrow"
].iloc[0]




In [ ]:
embedding_cols = [c for c in song_embeddings.columns if c.startswith("embedding_")]

black_vec = black_swan[embedding_cols].values.reshape(1, -1)
fake_vec = fake_love[embedding_cols].values.reshape(1, -1)
tomorrow_vec = tomorrow[embedding_cols].values.reshape(1, -1)


In [ ]:
print(
    "Black Swan ↔ Fake Love:",
    cosine_similarity(black_vec, fake_vec)[0][0]
)

print(
    "Black Swan ↔ Tomorrow:",
    cosine_similarity(black_vec, tomorrow_vec)[0][0]
)

## Top 10 msot similar songs to Black Swan

In [ ]:
embedding_cols = [c for c in song_embeddings.columns if c.startswith("embedding_")]

target_song = "Tomorrow"

target_vector = song_embeddings.loc[
    song_embeddings["track_name"] == target_song, 
    embedding_cols
].values

similarities = cosine_similarity(
    target_vector, 
    song_embeddings[embedding_cols]
)[0]

results = song_embeddings[[
    "track_name",
    "album_name", 
    "release_year"
]].copy()

results["similarity"] = similarities

results = results.sort_values(
    "similarity", 
    ascending=False
)

results.head(10)